# اجرای پروژه mT5 فارسی در Colab

**قبل از شروع:** منوی **Runtime → Change runtime type → T4 GPU**

- کد و داده از گیت‌هاب خوانده می‌شوند.
- **مدل‌ها و نتایج در Google Drive** ذخیره می‌شوند (پوشه `mt5_outputs`)؛ اگر Colab قطع شد چیزی از دست نمی‌رود.
- اگر Colab قطع شد: دوباره **Run all** بزنید. مرحله‌هایی که مدلشان در Drive هست **خودکار رد می‌شوند** و کار از مرحله بعد ادامه پیدا می‌کند.
- حجم لازم در Drive: حدود ۷ گیگابایت (۳ مدل × ۲.۳ گیگابایت).

## ۱. اتصال Drive، دریافت کد و داده

In [ ]:
import os, glob, zipfile, shutil
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/mt5_outputs'          # models and results are kept here
os.makedirs(OUT, exist_ok=True)

REPO = '/content/persian-mt5-summarization-sentiment'
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/zph272/persian-mt5-summarization-sentiment.git {REPO}
%cd {REPO}
!git pull -q

if not os.path.islink('outputs'):
    shutil.rmtree('outputs', ignore_errors=True)
    os.symlink(OUT, 'outputs')                       # outputs/ -> Google Drive

os.makedirs('data/processed', exist_ok=True)
for zp in glob.glob('*.zip'):
    zipfile.ZipFile(zp).extractall('data/processed')
if not os.path.exists('data/processed/joint_clean.csv'):
    shutil.copy('data/processed/joint_train.csv', 'data/processed/joint_clean.csv')
print(sorted(os.listdir('data/processed')))
print('Models already in Drive:', os.listdir(f'{OUT}/models') if os.path.isdir(f'{OUT}/models') else 'none')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## ۲. نصب کتابخانه‌ها

In [ ]:
!pip install -q rouge_score sentencepiece

## ۳. مرحله ۱ — خلاصه‌سازی (حدود ۳ ساعت)

In [ ]:
if os.path.isdir('outputs/models/summ_final'):
    print('✅ Stage 1 model exists in Drive — skipped')
else:
    !python 4_train_stage1_summ.py

## ۴. مرحله ۲ — تحلیل احساس (حدود ۳۰ دقیقه)

In [ ]:
if os.path.isdir('outputs/models/sent_final'):
    print('✅ Stage 2 model exists in Drive — skipped')
else:
    !python 5_train_stage2_sent.py

## ۵. مرحله ۳ — آموزش مشترک (حدود ۲ ساعت)

In [ ]:
if os.path.isdir('outputs/models/joint_final'):
    print('✅ Stage 3 model exists in Drive — skipped')
else:
    !python 6_train_stage3_joint.py

## ۶. ارزیابی روی داده تست
هر ارزیابی خلاصه‌سازی روی ۵٬۵۹۳ خبر حدود ۳۰ تا ۶۰ دقیقه است. ارزیابی‌های انجام‌شده رد می‌شوند.

In [ ]:
def done(name, key):
    import json
    p = f'outputs/results/{name}_eval.json'
    return os.path.exists(p) and key in json.load(open(p, encoding='utf-8'))

if not done('baselines', 'majority_sentiment'):
    !python 7_evaluate.py --baselines
if not done('sent_final', 'sentiment'):
    !python 7_evaluate.py --model sent_final --tasks sent
if not done('joint_final', 'summarization'):
    !python 7_evaluate.py --model joint_final --tasks summ,sent,loss
if not done('summ_final', 'summarization'):
    !python 7_evaluate.py --model summ_final --tasks summ,loss

## ۷. Ablation (اختیاری، در یک نشست دیگر)
- `sent_only`: فقط احساس از mT5-base (حدود ۳۰ دقیقه)
- `joint_only`: آموزش مشترک بدون مراحل ۱ و ۲ (حدود ۳ ساعت)

In [ ]:
RUN_ABLATION = False
if RUN_ABLATION:
    if not os.path.isdir('outputs/models/sent_only'):
        !python 9_ablation.py --only sent
    if not os.path.isdir('outputs/models/joint_only'):
        !python 9_ablation.py --only joint

## ۸. نمودارها و جدول نتایج

In [ ]:
!python 8_plot_results.py
from IPython.display import Image, display, Markdown
if os.path.exists('outputs/results/results_table.md'):
    display(Markdown(open('outputs/results/results_table.md', encoding='utf-8').read()))
for f in sorted(glob.glob('outputs/figures/*.png')):
    print(f); display(Image(f, width=600))

## ۹. فایل نتایج
نتایج در Drive هستند (`mt5_outputs/results` و `mt5_outputs/figures`). این سلول یک zip هم دانلود می‌کند.

In [ ]:
!cd outputs && zip -qr /content/results.zip results figures
from google.colab import files
files.download('/content/results.zip')